In [15]:
pip show pip

Name: pip
Version: 25.3
Summary: The PyPA recommended tool for installing Python packages.
Home-page: https://pip.pypa.io/
Author: 
Author-email: The pip developers <distutils-sig@python.org>
License-Expression: MIT
Location: /home/vmc/Source/wptPiFilter/.venv/lib/python3.11/site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install NetfilterQueue scapy

Looking in indexes: https://pypi.org/simple, https://www.piwheels.org/simple
Note: you may need to restart the kernel to use updated packages.


In [10]:
from platform import python_version
print(python_version())
!python --version
!python -V
!which python
!which pip

3.11.2
Python 3.11.2
Python 3.11.2
/home/vmc/Source/wptPiFilter/.venv/bin/python
/home/vmc/Source/wptPiFilter/.venv/bin/pip


In [9]:
pip list

Package                   Version
------------------------- -----------
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.1.0
attrs                     25.4.0
babel                     2.17.0
beautifulsoup4            4.14.3
bleach                    6.3.0
certifi                   2026.1.4
cffi                      2.0.0
charset-normalizer        3.4.4
comm                      0.2.3
debugpy                   1.8.19
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.21.2
fqdn                      1.5.1
h11                       0.16.0
httpcore                  1.0.9
httpx                     0.28.1
idna                      3.11
ipykernel                 7.1.0
ipython                   9.9.0
ipython_pygments_lexers   1.1.1
ipywidgets                8.1.8
isoduration         

In [1]:
# ~/nfq_filter.py
from netfilterqueue import NetfilterQueue
from scapy.all import IP, UDP, TCP, raw

PORT = 24680

def handle(pkt):
    ip = IP(pkt.get_payload())

    # UDP example: replace a string in payload
    if UDP in ip and (ip[UDP].dport == PORT or ip[UDP].sport == PORT):
        pld = bytes(ip[UDP].payload)
        new_pld = pld.replace(b"OLD", b"NEW")
        if new_pld != pld:
            ip[UDP].remove_payload()
            ip[UDP].add_payload(new_pld)
            # let Scapy recalc lengths/checksums
            del ip.len, ip.chksum, ip[UDP].len, ip[UDP].chksum
            pkt.set_payload(raw(ip))

    # For TCP, you can modify payload too, but be mindful of sequence/segmentation.
    pkt.accept()

if __name__ == "__main__":
    nfq = NetfilterQueue()
    nfq.bind(0, handle)
    try:
        nfq.run()
    finally:
        nfq.unbind()


KeyboardInterrupt: 

In [2]:

import socket
import threading
import struct
import ctypes

IP_TRANSPARENT = 19
SO_ORIGINAL_DST = 80  # Magic Linux constant

LISTEN_PORT = 19000
BUFFER = 65535

def get_original_dst(conn):
    # getsockopt(SO_ORIGINAL_DST) returns sockaddr_in (16 bytes)
    data = conn.getsockopt(socket.SOL_IP, SO_ORIGINAL_DST, 16)
    family, port, addr = struct.unpack("!HH4s8x", data)
    ip = socket.inet_ntoa(addr)
    return ip, port

def handle_client(client_sock):
    orig_ip, orig_port = get_original_dst(client_sock)
    print(f"[+] Original destination: {orig_ip}:{orig_port}")

    # Connect to real server
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.connect((orig_ip, orig_port))

    # Client->Server forwarder (with modification)
    def client_to_server():
        while True:
            data = client_sock.recv(BUFFER)
            if not data: 
                server.close()
                break

            # *** MODIFY PAYLOAD HERE ***
            modified = data.replace(b"OLD", b"NEW")

            # Or drop entirely:
            # continue

            # Or generate new payload:
            # modified = b"HELLO_NEW_PACKET"

            server.sendall(modified)

    # Server->Client forwarder (optional modification)
    def server_to_client():
        while True:
            data = server.recv(BUFFER)
            if not data:
                client_sock.close()
                break
            
            # optional reverse transformation
            client_sock.sendall(data)

    threading.Thread(target=client_to_server).start()
    threading.Thread(target=server_to_client).start()

def main():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_IP, IP_TRANSPARENT, 1)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("0.0.0.0", LISTEN_PORT))
    s.listen(128)

    print(f"[+] Transparent proxy listening on {LISTEN_PORT}")

    while True:
        client, addr = s.accept()
        print("[+] Connection from", addr)
        threading.Thread(target=handle_client, args=(client,)).start()

if __name__ == "__main__":
    main()


[+] Transparent proxy listening on 19000
[+] Connection from ('10.42.0.154', 39410)
[+] Original destination: 192.168.1.82:24680


Exception in thread Thread-6 (client_to_server):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7952/1561472642.py", line 30, in client_to_server
ConnectionResetError: [Errno 104] Connection reset by peer


[+] Connection from ('10.42.0.154', 47400)
[+] Original destination: 192.168.1.82:24680
[+] Connection from ('10.42.0.154', 49016)
[+] Original destination: 192.168.1.82:24680


Exception in thread Thread-12 (client_to_server):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_7952/1561472642.py", line 30, in client_to_server
ConnectionResetError: [Errno 104] Connection reset by peer


[+] Connection from ('10.42.0.154', 37082)
[+] Original destination: 192.168.1.82:24680


KeyboardInterrupt: 

In [15]:

#!/usr/bin/env python3
import socket
import struct

# Constants from Linux headers
IP_TRANSPARENT = 19             # setsockopt() option
SO_ORIGINAL_DST = 80            # get original destination
LISTEN_PORT = 19001             # must match your nft tproxy rule
BUFFER = 65535

def get_original_dst(sock):
    """
    Reads Linux SO_ORIGINAL_DST metadata to learn the real destination
    of the packet intercepted by TPROXY.
    Return (ip, port).
    """
    data = sock.getsockopt(socket.SOL_IP, SO_ORIGINAL_DST, 16)
    # struct sockaddr_in = {sin_family (2 bytes), port (2), addr (4), padding (8)}
    _, port, raw_ip = struct.unpack("!HH4s8x", data)
    ip = socket.inet_ntoa(raw_ip)
    return ip, port

def main():
    # Create the transparent UDP receiving socket
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
    s.setsockopt(socket.SOL_IP, IP_TRANSPARENT, 1)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    # Bind to 0.0.0.0:19001 (your tproxy target)
    s.bind(("0.0.0.0", LISTEN_PORT))

    print(f"[+] UDP Transparent Proxy listening on port {LISTEN_PORT}")

    while True:
        data, ancdata, flags, client_addr = s.recvmsg(BUFFER, 1024)
        #data, client_addr = s.recvfrom(BUFFER)

        print(f"[+] Received UDP {len(data)} bytes from {client_addr}")

        
        orig_ip = None
        orig_port = None

        # Extract original destination from ancillary data
        for cmsg_level, cmsg_type, cmsg_data in ancdata:
            if cmsg_level == socket.SOL_IP and cmsg_type == SO_ORIGINAL_DST:
                family, port, raw_ip = struct.unpack("!HH4s8x", cmsg_data)
                orig_ip = socket.inet_ntoa(raw_ip)
                orig_port = port

        
        if not orig_ip:
            print("[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.")
            continue


        # Get the ORIGINAL destination (IP:Port)
        #orig_ip, orig_port = get_original_dst(s)
        print(f"    Original destination was {orig_ip}:{orig_port}")

        # ---------------------------------------------------------------------
        # *** MODIFY / FILTER / REWRITE THE PACKET HERE ***
        # ---------------------------------------------------------------------

        # Example 1: drop packet containing pattern
        if b"BLOCK" in data:
            print("    Dropped packet.")
            continue

        # Example 2: simple byte replacement
        new_data = data.replace(b"OLD", b"NEW")

        # Example 3: completely new synthetic packet:
        # new_data = b"THIS IS A NEW UDP PACKET"

        # ---------------------------------------------------------------------
        # Forward modified packet to original destination
        # ---------------------------------------------------------------------

        upstream = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        # If you want to spoof the client's source address AND port,
        # you must set IP_TRANSPARENT and bind to client_addr before sending.
        # Example:
        # upstream.setsockopt(socket.SOL_IP, IP_TRANSPARENT, 1)
        # upstream.bind(client_addr)

        upstream.sendto(new_data, (orig_ip, orig_port))
        upstream.close()

        print(f"    Forwarded {len(new_data)} bytes to {orig_ip}:{orig_port}")

        # ---------------------------------------------------------------------
        # If you also want to intercept *server → client* return datagrams,
        # you need matching TPROXY rules for packets coming back from eth0.
        # ---------------------------------------------------------------------

if __name__ == "__main__":
    main()


[+] UDP Transparent Proxy listening on port 19001
[+] Received UDP 16 bytes from ('10.42.0.154', 46578)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 47035)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 48960)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 37518)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 39993)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 40374)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 47558)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST.
[+] Received UDP 16 bytes from ('10.42.0.154', 43832)
[-] Packet was NOT delivered through TPROXY. No SO_ORIGINAL_DST

KeyboardInterrupt: 

In [11]:
import socket
import struct

# Constants from Linux headers
IP_TRANSPARENT = 19             # setsockopt() option
SO_ORIGINAL_DST = 80            # get original destination
LISTEN_PORT = 19001             # must match your nft tproxy rule
BUFFER = 65535

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
s.setsockopt(socket.SOL_IP, IP_TRANSPARENT, 1)
s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# Bind to 0.0.0.0:19001 (your tproxy target)
s.bind(("0.0.0.0", LISTEN_PORT))
print("IP_TRANSPARENT:", s.getsockopt(socket.SOL_IP, 19))
data, ancdata, flags, client_addr = s.recvmsg(65535, 1024)

IP_TRANSPARENT: 1


In [ ]:
#!/usr/bin/env python3
import socket, struct

IP_TRANSPARENT       = 19            # SOL_IP level
IP_RECVORIGDSTADDR   = 20            # enables IP_ORIGDSTADDR CMSG on recvmsg()
IP_ORIGDSTADDR       = 20            # CMSG type carrying struct sockaddr_in
LISTEN_PORT          = 19001
BUF                  = 65535

def main():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM, socket.IPPROTO_UDP)
    s.setsockopt(socket.SOL_IP, IP_TRANSPARENT, 1)       # required for TPROXY
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.setsockopt(socket.SOL_IP, IP_RECVORIGDSTADDR, 1)   # <-- crucial for UDP
    s.bind(("0.0.0.0", LISTEN_PORT))

    print(f"[+] Transparent UDP proxy listening on {LISTEN_PORT}")

    while True:
        data, ancdata, flags, client_addr = s.recvmsg(BUF, 1024)
        print(f"[+] Received {len(data)} bytes from {client_addr}")

        orig_ip, orig_port = None, None
        for level, ctype, cdata in ancdata:
            if level == socket.SOL_IP and ctype == IP_ORIGDSTADDR:
                # struct sockaddr_in: family(2) port(2) addr(4) + 8 pad
                family, port, raw_ip = struct.unpack("!HH4s8x", cdata)
                orig_ip = socket.inet_ntoa(raw_ip)
                orig_port = port
                break

        if not orig_ip:
            print("[-] Packet was NOT delivered through UDP ORIGDSTADDR CMSG.")
            continue

        print(f"    Original destination = {orig_ip}:{orig_port}")

        # ---- MODIFY / FILTER HERE ----
        new_data = data.replace(b"OLD", b"NEW")
        # --------------------------------

        out = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        out.sendto(new_data, (orig_ip, orig_port))
        out.close()
        print(f"    Forwarded {len(new_data)} bytes to server.")

if __name__ == "__main__":
    main()


[+] Transparent UDP proxy listening on 19001
[+] Received 16 bytes from ('10.42.0.154', 38966)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
[+] Received 16 bytes from ('10.42.0.154', 44833)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
[+] Received 16 bytes from ('10.42.0.154', 47343)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
[+] Received 16 bytes from ('10.42.0.154', 45666)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
[+] Received 16 bytes from ('10.42.0.154', 46163)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
[+] Received 16 bytes from ('10.42.0.154', 44360)
    Original destination = 192.168.1.82:24680
    Forwarded 16 bytes to server.
